# OECD Export Restrictions Database: Cleaning and Analysis

**Analyst's brief:** this is the ground-truth events dataset behind an early-warning model for food export restrictions on Wheat and Rice, built for the Women in Data "What's Cooking?" Datathon. A predictive model is only as good as its labels, so this notebook's job is to turn a raw administrative database into a verified, defensible set of restriction events, and to extract every pattern in it worth reporting before any trade or production data is even touched.

**Source:** OECD, Export Restrictions Database for Staple Crops (2025), sheet `DetailedDatabase`, compiled under the G20 Agricultural Market Information System (AMIS). AMIS members cover 94 to 97 percent of world trade in wheat, rice, maize, and soybeans, and only restrictions that were actually implemented or officially announced in a legal document are included.


## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 100)


## 1. Load and scope

**File needed:** `oecd-database-on-export-restrictions-for-staple-crops-2025.xlsx`

The raw file covers six commodity classes and six measure types. This project uses two commodities, Wheat and Rice, and three measure types: Export prohibition, Export quota, and Minimum export price. Export tax and Licensing requirements are left out on purpose. A tax raises cost without capping volume, and a licensing requirement is closer to a paperwork step than a real restriction. Neither belongs in the same category as a measure that actually blocks or caps trade.

In [ ]:
df = pd.read_excel("oecd-database-on-export-restrictions-for-staple-crops-2025.xlsx", sheet_name="DetailedDatabase")
print("Total rows in the raw file:", len(df))

broad_measures = ["Export prohibition", "Export quota", "Minimum export price"]
sub = df[
    (df["CommodityClass_Name"].isin(["Wheat", "Rice"]))
    & (df["PolicyMeasure_Name"].isin(broad_measures))
].copy()

print("Rows after scoping to Wheat, Rice, and the three measure types:", len(sub))


## 2. Clean: collapse customs-code duplication

**Problem:** a single restriction decision is frequently split across several rows in this file, one per customs tariff line it covers. Left as is, one real policy would be counted several times, inflating every count that follows.

**Fix:** group by Country, Commodity, and Start Date, the three fields that together identify one real government decision. When a group's rows disagree on measure type, which happens, keep the most severe measure (prohibition over quota over minimum price), rather than whichever row the spreadsheet happened to list first.

In [ ]:
severity_rank = {"Export prohibition": 3, "Export quota": 2, "Minimum export price": 1}
sub["Severity_Rank"] = sub["PolicyMeasure_Name"].map(severity_rank)
sub_sorted = sub.sort_values("Severity_Rank", ascending=False)

events = sub_sorted.groupby(
    ["Country_Name", "CommodityClass_Name", "Start_Date"], as_index=False
).agg({
    "End_Date": "max",
    "PolicyMeasure_Name": "first",
    "Short_Description": "first",
    "Product_Original_Name": "first",
    "Source": "first",
})
events["Duration_Days"] = (events["End_Date"] - events["Start_Date"]).dt.days
events["Start_Year"] = events["Start_Date"].dt.year

print("Real, distinct events after cleaning:", len(events))
print(events["CommodityClass_Name"].value_counts())


**Opinion:** 494 raw rows collapse to 209 real events, a reduction of more than half. Any report that quotes "494 restriction events" from this file without doing this step first is materially wrong, not merely imprecise. This is the single most consequential cleaning step in the whole notebook.

## 3. Verify scope: raw grain, or a processed derivative

**Problem:** the label "Wheat" or "Rice" on an event does not guarantee the restriction covers raw grain. It can legally cover flour, groats, starch, gluten, or bran under a different customs code, filed under the same commodity name. A model built on raw-grain trade data has no way to explain a restriction that was actually about flour.

**Method, and a real mistake worth stating plainly:** the first version of this check looked at the numeric HS code, and where that was blank, fell back to the product name text. That first version had a bug: when the product-name field itself was blank, concatenating it with the description silently produced a missing value instead of an empty string, which meant the text search was never actually run for a meaningful number of events. Fixing that (filling blanks with an empty string before concatenating, not after) changed the result materially for Rice: fifteen Rice events, all from China, were only caught as genuine rice-flour and rice-groats restrictions once this fix was in place. This is a useful lesson on its own: a text-matching data quality check needs to be tested against its own edge cases (blank fields) as carefully as the data itself.

In [ ]:
raw_codes = {"Rice": "1006", "Wheat": "1001"}
processed_codes = {
    "Rice": ["1101", "1102", "1103", "1108", "1109", "1904", "2302"],
    "Wheat": ["1101", "1103", "1108", "1109", "1904", "2302"],
}
processed_keywords = ["flour", "groats", "meal", "starch", "bran", "gluten", "pellet"]

def classify_scope(commodity, country, start_date):
    rows = df[
        (df["CommodityClass_Name"] == commodity)
        & (df["Country_Name"] == country)
        & (df["Start_Date"] == start_date)
        & (df["PolicyMeasure_Name"].isin(broad_measures))
    ]
    hs4 = rows["HS_Code"].astype(str).str[:4]

    if raw_codes[commodity] in hs4.values:
        return "raw"
    if hs4.isin(processed_codes[commodity]).any():
        return "processed_only"

    # Fill blanks BEFORE concatenating, so a missing product name cannot
    # silently erase a usable description.
    text = (
        rows["Product_Original_Name"].fillna("").astype(str) + " "
        + rows["Short_Description"].fillna("").astype(str)
    ).str.lower()
    if text.str.contains("|".join(processed_keywords)).any():
        return "processed_only"
    return "raw_unflagged"

events["Scope"] = events.apply(lambda r: classify_scope(r["CommodityClass_Name"], r["Country_Name"], r["Start_Date"]), axis=1)
print(events.groupby("CommodityClass_Name")["Scope"].value_counts())


In [ ]:
events_for_label = events[events["Scope"] != "processed_only"].copy()
excluded = events[events["Scope"] == "processed_only"]

print("Excluded as confirmed processed-product restrictions:")
print(excluded[["Country_Name", "CommodityClass_Name", "Start_Date", "Short_Description"]].to_string())
print()
print("Usable events retained, by commodity:")
print(events_for_label["CommodityClass_Name"].value_counts())


**Opinion:** 29 of 209 events, about 14 percent, are confirmed restrictions on a processed product, not raw grain, and all 29 are excluded from the label used later. Wheat is the more affected commodity proportionally (14 of 67, 21 percent), concentrated in India's repeated flour restrictions across four separate crisis periods and Kazakhstan and Russia's 2020 groats restrictions. Rice's 15 excluded events are entirely China's rice flour and groats quotas, a single country's pattern, not a broad one. India and Vietnam, Rice's two largest restrictors, have zero events excluded on this basis, meaning this correction does not touch the countries the Rice model will lean on most.

---
## 4. Decide country scope from the evidence, before any trade data is touched

A country with no restriction history for a given commodity can never supply a real positive example for that commodity. Scope has to be read off this data now, not assumed from convenience, and not discovered as a problem later.

In [ ]:
rice = events_for_label[events_for_label["CommodityClass_Name"] == "Rice"]
wheat = events_for_label[events_for_label["CommodityClass_Name"] == "Wheat"]

print("Countries with a real Rice restriction history:")
print(rice["Country_Name"].value_counts())
print()
print("Countries with a real Wheat restriction history:")
print(wheat["Country_Name"].value_counts())


**Opinion:** the two lists share only three countries (India, Russia, Egypt), out of a combined eight distinct countries. China restricts rice (in its processed form only, per Section 3) but never appears as a raw-wheat or raw-rice restrictor once flour is excluded. Vietnam restricts rice but never wheat. Ukraine and Kazakhstan restrict wheat but never rice. Wheat and Rice are two separate restriction problems that happen to share a template, not one generic staple-crop pattern, and any trade data pulled later has to be scoped separately for each.

---
## 5. Frequency: concentration of restriction behavior

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
rice["Country_Name"].value_counts().sort_values().plot(kind="barh", ax=axes[0], color="tab:green")
axes[0].set_title("Rice: Restriction Events by Country (raw grain only)")
wheat["Country_Name"].value_counts().sort_values().plot(kind="barh", ax=axes[1], color="tab:blue")
axes[1].set_title("Wheat: Restriction Events by Country (raw grain only)")
plt.tight_layout()
plt.show()


**Opinion:** India is the single most important country in this dataset, accounting for 55 of 127 usable Rice events (43 percent) and 13 of 53 usable Wheat events (25 percent). No other country comes close on the Rice side; Vietnam is a distant second at 45 events (35 percent), and every other Rice-restricting country combined accounts for barely a fifth of the total. Wheat's distribution is flatter and more genuinely multi-country, led by Ukraine (15) with India, Argentina, and Russia all in a similar range behind it. A practical consequence: any validation of a Rice model will be, to a significant degree, a validation of how well it predicts India and Vietnam specifically, and that should be stated as a limitation, not glossed over.

## 6. Timing: crisis-driven, not random

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
rice["Start_Year"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="tab:orange")
axes[0].set_title("Rice Restrictions, by Year")
wheat["Start_Year"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="tab:red")
axes[1].set_title("Wheat Restrictions, by Year")
plt.tight_layout()
plt.show()


**Opinion:** this is the strongest single finding in the dataset. Rice's clear peak is 2008, the global food price crisis, with 20 events in that year alone, roughly one in six of every Rice event in the dataset. Wheat's response to that same year is much smaller (3 events); its own peak years are 2011 (9 events, Russia's drought) and 2022 (8 events, the Russia-Ukraine war), years where Rice's activity is comparatively muted. These are not the same crisis producing the same reaction twice, they are different shocks producing different, commodity-specific responses, and that is direct evidence that a single blended staple-crop risk score would blur a real, learnable distinction that a commodity-specific model can capture.

## 7. Measure type: different policy instruments, by commodity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
rice_measure = rice["PolicyMeasure_Name"].value_counts()
axes[0].pie(rice_measure, labels=rice_measure.index, autopct="%1.0f%%")
axes[0].set_title("Rice: Type of Restriction")
wheat_measure = wheat["PolicyMeasure_Name"].value_counts()
axes[1].pie(wheat_measure, labels=wheat_measure.index, autopct="%1.0f%%")
axes[1].set_title("Wheat: Type of Restriction")
plt.tight_layout()
plt.show()


**Opinion:** Wheat restrictions are dominated by export quotas (59 percent) and rarely use a minimum price floor (6 percent). Rice restrictions are led by minimum export price (40 percent), with quota and prohibition close behind. This is a genuine behavioral difference, not noise: Wheat's largest restrictors (Ukraine, Russia, Argentina, all large-scale grain exporters) tend toward a blunt volume cap, while Rice's largest restrictors (India, Vietnam) use a price floor more often, a tool that protects farmer income without stopping trade outright. This is a reasoned interpretation consistent with the pattern in this dataset; it is not a mechanism this dataset alone can prove, and should be presented as informed judgment, not as an established fact.

## 8. Duration: how long a restriction actually lasts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].hist(rice["Duration_Days"].dropna(), bins=20, color="tab:purple", edgecolor="white")
axes[0].set_title("Rice: Restriction Duration (Days)")
axes[1].hist(wheat["Duration_Days"].dropna(), bins=20, color="tab:brown", edgecolor="white")
axes[1].set_title("Wheat: Restriction Duration (Days)")
plt.tight_layout()
plt.show()
print("Median duration, Rice:", rice["Duration_Days"].median(), "days, n =", rice["Duration_Days"].notna().sum())
print("Median duration, Wheat:", wheat["Duration_Days"].median(), "days, n =", wheat["Duration_Days"].notna().sum())


**Opinion:** Rice's typical restriction (median 151 days, about five months) is somewhat shorter than Wheat's (median 176 days, about six months), and this is directionally consistent with the measure-type finding above: a price floor is a lighter-touch tool that a government can lift sooner, while a quota is often part of a more sustained policy stance. Both distributions have a real tail past 1,000 days; Rice's longest recorded restriction ran 1,489 days, roughly four years. A restriction lasting four years is a structurally different kind of event from one lasting a few weeks, and averaging them into one number understates that range. This is worth naming individually in a presentation rather than folding into a single "typical duration" statistic.

---
## 9. Summary table and closing assessment

In [ ]:
summary = events_for_label.groupby(["CommodityClass_Name", "Country_Name"]).agg(
    total_events=("Start_Date", "count"),
    median_duration_days=("Duration_Days", "median"),
).reset_index().sort_values(["CommodityClass_Name", "total_events"], ascending=[True, False])
print(summary.to_string())

events_for_label.to_csv("events_clean_final.csv", index=False)
from google.colab import files
files.download("events_clean_final.csv")


**Closing assessment.** This dataset is usable, but not out of the box. Naive use of the raw file overstates event count by more than half, and a naive scope check, done without testing its own edge cases, would have quietly kept 15 confirmed rice-flour restrictions labeled as raw rice. After both corrections, 180 of 209 events (86 percent) are retained as genuine raw-grain restrictions, split 127 Rice and 53 Wheat, across eight countries whose overlap between the two commodities is smaller than a convenience-based country list would assume.

The single most citable finding is Section 6: three different global food crises produced three commodity-specific restriction signatures, not one repeating pattern. That is the direct justification for building separate, commodity-specific models rather than one generic staple-crop risk score, and it is the finding this presentation should open with.

What this dataset cannot do: it carries no information on why one measure was chosen over another beyond what is offered here as informed interpretation, no information on whether a restriction achieved its goal, and no economic data of its own. It is a clean, verified label set. Every "why" in this notebook is reasoned judgment consistent with the pattern, not a claim this dataset independently proves, and the write-up should say so plainly rather than overstate it.
